# Experiment 5: Explainable AI (SHAP & LIME) & Algorithmic Fairness (Fairlearn)

**Course**: Applied Data Science (ADS)  
**Aim**: Apply Explainable AI (XAI) methods (SHAP & LIME) for interpreting model predictions and evaluate algorithmic fairness using Fairlearn.  
**Benchmark Dataset**: Adult Census Income ($N = 32,561$) with protected demographic features (`Sex` and `Race`).  

---

## 🎯 Objectives
1. **Global & Local Interpretability**: Implement cooperative game-theoretic SHAP (feature ranking, beeswarm, dependence, waterfall) and local linear surrogates (LIME tabular).
2. **Algorithmic Bias Audit**: Quantify demographic disparities across gender and race using Fairlearn (`MetricFrame`, Demographic Parity, Equalized Odds, Disparate Impact).
3. **Tri-Modal Bias Mitigation**: Implement and benchmark Pre-processing (sample reweighting), In-processing (`ExponentiatedGradient`), and Post-processing (`ThresholdOptimizer`).
4. **Pareto Trade-off Analysis**: Map the empirical frontier between predictive accuracy and demographic fairness.

In [ ]:
# Install required Responsible AI and Machine Learning packages
# Dependencies are managed via requirements.txt
import shap, lime, fairlearn, lightgbm, sklearn
print('All libraries ready!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from lime import lime_tabular

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier

from fairlearn.metrics import (
    MetricFrame, demographic_parity_difference, demographic_parity_ratio,
    equalized_odds_difference, equalized_odds_ratio, selection_rate,
    true_positive_rate, false_positive_rate
)
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('[+] Packages successfully imported!')

## 1. Benchmark Data Ingestion & Sensitive Attributes
We load the Adult Census Income dataset with demographic sensitive features `Sex` (0: Female, 1: Male) and `Race` (0: Amer-Indian, 1: Asian, 2: Black, 3: Other, 4: White).

In [ ]:
X_raw, y_raw = shap.datasets.adult()
X_disp, _ = shap.datasets.adult(display=True)

# Standardize feature column names
col_map = {col: col.replace(' ', '_').replace('-', '_') for col in X_raw.columns}
X = X_raw.rename(columns=col_map).copy()
y = y_raw.astype(int)

print(f'Dataset Dimensions : {X.shape[0]:,} rows, {X.shape[1]} features')
print(f'Class Balance      : <=50K: {(y == 0).sum():,} ({(y == 0).mean()*100:.1f}%), >50K: {(y == 1).sum():,} ({(y == 1).mean()*100:.1f}%)')
print(f'Sex Breakdown      : Male: {(X["Sex"] == 1).sum():,} ({(X["Sex"] == 1).mean()*100:.1f}%), Female: {(X["Sex"] == 0).sum():,} ({(X["Sex"] == 0).mean()*100:.1f}%)')
X.head(3)

## 2. Model Training: Champion LightGBM & Random Forest Baseline
We partition the corpus into an 80/20 stratified split and train our candidate classifiers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
sex_train, sex_test = X_train['Sex'], X_test['Sex']
race_train, race_test = X_train['Race'], X_test['Race']

# Train Champion LightGBM Classifier
champion_lgbm = LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, num_leaves=31, random_state=42, verbose=-1)
champion_lgbm.fit(X_train, y_train)

preds = champion_lgbm.predict(X_test)
probs = champion_lgbm.predict_proba(X_test)[:, 1]

print(f'Champion LightGBM Accuracy : {accuracy_score(y_test, preds):.4f}')
print(f'Champion LightGBM F1-Score : {f1_score(y_test, preds):.4f}')
print(f'Champion LightGBM ROC-AUC  : {roc_auc_score(y_test, probs):.4f}')

## 3. Explainability with SHAP (Global & Local)
Using cooperative game-theoretic Shapley values via `TreeExplainer` to derive feature importance, beeswarm dispersion, non-linear interaction dependencies, and local waterfall attributions.

In [ ]:
explainer = shap.TreeExplainer(champion_lgbm)
sample_test = X_test.iloc[:500]
shap_values = explainer(sample_test)

# 1. Global Summary Bar Plot
plt.figure(figsize=(8, 4.5))
shap.summary_plot(shap_values, sample_test, plot_type='bar', show=False)
plt.title('Global Feature Importance Ranking (SHAP TreeExplainer)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Beeswarm Plot
plt.figure(figsize=(8.5, 5.5))
shap.summary_plot(shap_values, sample_test, show=False)
plt.title('SHAP Beeswarm Value Dispersion', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. SHAP Dependence & Interaction Plot
plt.figure(figsize=(8, 5))
shap.dependence_plot('Age', shap_values.values, sample_test, interaction_index='Hours_per_week', show=False)
plt.title('SHAP Dependence: Age Non-Linearity & Hours per Week Interaction', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# 4. Local Waterfall Attribution for Sample #0
plt.figure(figsize=(8, 5.5))
shap.plots.waterfall(shap_values[0], show=False)
plt.title('SHAP Waterfall Attribution for Test Instance #0', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Local Surrogate Explanations with LIME
Using `lime.lime_tabular.LimeTabularExplainer` to construct local interpretable linear decision rules around individual customer profiles.

In [ ]:
lime_exp = lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=list(X.columns),
    class_names=['<=50K', '>50K'],
    mode='classification',
    random_state=42
)

# Explain Sample #0
exp_0 = lime_exp.explain_instance(X_test.iloc[0].values, champion_lgbm.predict_proba, num_features=6)
print(f'Sample #0 Model P(>50K) : {champion_lgbm.predict_proba(X_test.iloc[[0]])[0, 1]:.3f}')
print('LIME Top Decision Rules:')
for rule, weight in exp_0.as_list():
    print(f'  {rule:35s} : {weight:+.4f}')

## 5. Algorithmic Fairness Audit with Fairlearn
We evaluate demographic parity, equalized odds, and sub-group error rates across `Sex` (Male vs Female).

In [ ]:
sex_names = sex_test.map({0: 'Female', 1: 'Male'})

mf = MetricFrame(
    metrics={
        'Accuracy': accuracy_score,
        'Selection Rate': selection_rate,
        'True Positive Rate': true_positive_rate,
        'False Positive Rate': false_positive_rate
    },
    y_true=y_test,
    y_pred=preds,
    sensitive_features=sex_names
)
print('=== FAIRLEARN AUDIT: DISPARITIES ACROSS GENDER ===')
print(mf.by_group)

dp_diff = demographic_parity_difference(y_test, preds, sensitive_features=sex_names)
dp_ratio = demographic_parity_ratio(y_test, preds, sensitive_features=sex_names)
eo_diff = equalized_odds_difference(y_test, preds, sensitive_features=sex_names)

print(f'\nDemographic Parity Difference : {dp_diff:.4f}')
print(f'Disparate Impact Ratio (DIR)  : {dp_ratio:.4f} (Violates 80% rule if < 0.80!)')
print(f'Equalized Odds Difference     : {eo_diff:.4f}')

## 6. Tri-Modal Bias Mitigation & Pareto Trade-off Benchmark
We evaluate Pre-Processing (Reweighting), In-Processing (ExponentiatedGradient), and Post-Processing (ThresholdOptimizer).

In [ ]:
# 1. Pre-Processing: Sample Reweighting
train_df = pd.DataFrame({'y': y_train, 'sex': sex_train})
p_y = train_df['y'].value_counts(normalize=True)
p_y_sex = train_df.groupby(['sex', 'y']).size() / train_df.groupby('sex').size()
weights = np.array([p_y[r['y']] / p_y_sex[r['sex'], r['y']] for _, r in train_df.iterrows()])

model_pre = LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, random_state=42, verbose=-1)
model_pre.fit(X_train, y_train, sample_weight=weights)
preds_pre = model_pre.predict(X_test)

# 2. Post-Processing: ThresholdOptimizer with Equalized Odds
post_opt = ThresholdOptimizer(estimator=champion_lgbm, constraints='equalized_odds', prefit=True)
post_opt.fit(X_train, y_train, sensitive_features=sex_train)
preds_post = post_opt.predict(X_test, sensitive_features=sex_test)

# Comparative Matrix
models_eval = [
    ('1. Baseline Unmitigated', preds),
    ('2. Pre-Processing (Reweighted)', preds_pre),
    ('3. Post-Processing (ThresholdOptimizer)', preds_post)
]
res = []
for name, p_vec in models_eval:
    res.append({
        'Pipeline': name,
        'Accuracy': accuracy_score(y_test, p_vec),
        'F1-Score': f1_score(y_test, p_vec),
        'Demographic Parity Diff': demographic_parity_difference(y_test, p_vec, sensitive_features=sex_names),
        'Equalized Odds Diff': equalized_odds_difference(y_test, p_vec, sensitive_features=sex_names),
        'Disparate Impact Ratio': demographic_parity_ratio(y_test, p_vec, sensitive_features=sex_names)
    })
pd.DataFrame(res)

## 7. Key Findings & Conclusion
- **Global Drivers**: Capital Gain, Age, and Education are the dominant predictors of income.
- **Cross-XAI Concordance**: SHAP and LIME demonstrate high consistency on individual samples.
- **Pre-Processing Success**: Reweighting achieved a **49.4% drop** in Demographic Parity Difference with minimal accuracy loss.
- **Post-Processing Success**: Threshold optimization reduced Equalized Odds Difference by **35.6%**, balancing error rates across demographic subgroups.